# 🔍 Recherche Avancée - AskMe Search

Ce notebook permet d'explorer toutes les capacités de recherche avec le système de droits optionnels.

**Use cases :**
- 🔍 Recherche simple et avancée
- 🔐 Recherche avec filtrage par droits utilisateur
- 📊 Analyse des résultats de recherche
- 🎯 Recherche ciblée par type de contenu
- 📈 Métriques et performance
- 🧪 Tests de différents scénarios

## 📦 Configuration

In [ ]:
import sys
sys.path.append('../scripts')

from simple_indexer import SimpleIndexer
from pathlib import Path
import requests
import json
from typing import List, Dict, Optional
from datetime import datetime
import time

# Configuration
OPENSEARCH_URL = "http://localhost:9200"
CLIENTS_DATA_DIR = "../clients-data"

print("🔍 Notebook Recherche Avancée chargé")
print(f"   OpenSearch: {OPENSEARCH_URL}")
print(f"   Architecture: Multi-clients avec documents UUID + PJ multiples")

## 🔧 Configuration de l'Indexeur

In [ ]:
# Choisir l'index à utiliser
INDEX_NAME = DEFAULT_INDEX  # Changez ici pour un client: "askme-mon-client"

# Initialiser l'indexeur
indexer = SimpleIndexer(opensearch_url=OPENSEARCH_URL, index_name=INDEX_NAME)

print(f"🔧 Configuration de recherche:")
print(f"   Index cible: {INDEX_NAME}")

# Vérifier les données disponibles
stats = indexer.get_index_stats()
if stats:
    docs_count = stats.get('documents_count', 0)
    size_mb = stats.get('size_mb', 0)
    print(f"   📊 {docs_count} documents disponibles ({size_mb} MB)")
    
    if docs_count == 0:
        print(f"   ⚠️ Index vide - Indexez des documents d'abord avec le notebook 02")
    else:
        print(f"   ✅ Prêt pour la recherche !")
else:
    print(f"   ❌ Index introuvable ou erreur")
    print(f"   💡 Utilisez le notebook 01 pour créer un client ou l'index")

## 🔍 Recherche de Base

In [ ]:
def search_and_display(query: str, user_rights: List[str] = None, size: int = 5, 
                      show_details: bool = True) -> Dict:
    """Recherche avec affichage détaillé des résultats"""
    
    print(f"🔍 Recherche: '{query}'")
    if user_rights:
        rights_str = ", ".join(user_rights)
        print(f"👤 Droits utilisateur: {rights_str}")
    else:
        print(f"🌐 Recherche libre (tous documents)")
    
    print(f"📋 Limite: {size} résultats")
    print("=" * 70)
    
    # Mesurer le temps de recherche
    start_time = time.time()
    results = indexer.search(query, user_rights, size)
    search_time = time.time() - start_time
    
    if not results or 'hits' not in results:
        print(f"❌ Aucun résultat trouvé pour '{query}'")
        return {'query': query, 'total': 0, 'results': [], 'search_time': search_time}
    
    hits = results['hits']['hits']
    total = results['hits']['total']['value']
    
    print(f"📊 {total} résultats trouvés (affichage de {len(hits)})")
    print(f"⏱️ Temps de recherche: {search_time:.3f}s")
    print()
    
    processed_results = []
    
    for i, hit in enumerate(hits, 1):
        source = hit['_source']
        score = hit['_score']
        access_rights = source.get('accessRights', None)
        
        # Icône selon les droits
        if access_rights:
            rights_str = ", ".join(access_rights) if isinstance(access_rights, list) else str(access_rights)
            icon = "🔐"
            rights_display = f"Droits: {rights_str}"
        else:
            icon = "🌐"
            rights_display = "Accès libre"
        
        print(f"{i}. {icon} {source.get('title', 'Sans titre')} (score: {score:.2f})")
        print(f"   📂 {source.get('filepath', 'N/A')}")
        print(f"   🔑 {rights_display}")
        
        # Contenu avec surlignage
        if 'highlight' in hit and 'content' in hit['highlight']:
            content = hit['highlight']['content'][0]
            print(f"   💡 Extrait: {content}")
        else:
            content = source.get('content', '')[:150] + "..."
            print(f"   📄 Contenu: {content}")
        
        if show_details:
            chunk_id = source.get('chunk_id', 0)
            chunk_size = source.get('chunk_size', 0)
            print(f"   🧩 Chunk {chunk_id} ({chunk_size} caractères)")
        
        print()
        
        # Stocker pour l'analyse
        processed_results.append({
            'title': source.get('title', 'Sans titre'),
            'filepath': source.get('filepath', 'N/A'),
            'score': score,
            'access_rights': access_rights,
            'chunk_id': source.get('chunk_id', 0),
            'has_highlight': 'highlight' in hit
        })
    
    return {
        'query': query,
        'user_rights': user_rights,
        'total': total,
        'returned': len(hits),
        'search_time': search_time,
        'results': processed_results
    }

# Tests de recherche de base
print("🔍 Tests de recherche de base:")
print("=" * 50)

# Exemples de recherches (modifiez selon vos données)
test_queries = [
    "télétravail",
    "budget", 
    "configuration",
    "sécurité"
]

# Recherche simple
if test_queries:
    sample_query = test_queries[0]
    print(f"🧪 Test avec la requête: '{sample_query}'")
    result = search_and_display(sample_query, size=3)
    
    if result['total'] > 0:
        print(f"✅ Recherche fonctionnelle !")
    else:
        print(f"⚠️ Aucun résultat - Essayez d'autres termes ou indexez plus de documents")
else:
    print(f"💡 Modifiez les 'test_queries' selon vos documents indexés")

## 🔐 Recherche avec Droits d'Accès

In [ ]:
def compare_search_with_without_rights(query: str, user_rights: List[str]) -> Dict:
    """Comparer les résultats avec et sans droits"""
    
    print(f"🔍 Comparaison de recherche: '{query}'")
    print("=" * 60)
    
    # Recherche libre (tous documents)
    print(f"🌐 1. Recherche LIBRE (tous documents):")
    free_results = search_and_display(query, user_rights=None, size=5, show_details=False)
    
    print(f"\n🔐 2. Recherche avec DROITS {user_rights}:")
    filtered_results = search_and_display(query, user_rights=user_rights, size=5, show_details=False)
    
    # Analyse comparative
    print(f"\n📊 COMPARAISON:")
    print("=" * 40)
    print(f"🌐 Recherche libre:")
    print(f"   📄 Résultats: {free_results['total']}")
    print(f"   ⏱️ Temps: {free_results['search_time']:.3f}s")
    
    print(f"🔐 Recherche filtrée:")
    print(f"   📄 Résultats: {filtered_results['total']}")
    print(f"   ⏱️ Temps: {filtered_results['search_time']:.3f}s")
    
    # Calculer la différence
    diff = free_results['total'] - filtered_results['total']
    if diff > 0:
        print(f"🚫 Documents filtrés: {diff} ({diff/free_results['total']*100:.1f}%)")
    elif diff == 0:
        print(f"✅ Aucun document filtré (droits permettent tout)")
    else:
        print(f"⚠️ Anomalie: plus de résultats avec filtrage")
    
    return {
        'query': query,
        'user_rights': user_rights,
        'free_total': free_results['total'],
        'filtered_total': filtered_results['total'],
        'filtered_count': diff,
        'free_time': free_results['search_time'],
        'filtered_time': filtered_results['search_time']
    }

def test_different_user_profiles():
    """Tester différents profils utilisateur"""
    
    print(f"👥 Test de différents profils utilisateur:")
    print("=" * 60)
    
    # Profils de test
    user_profiles = [
        {
            'name': '🌐 Visiteur (libre)',
            'rights': None,
            'description': 'Accès à tous les documents libres uniquement'
        },
        {
            'name': '👤 Employé Finance',
            'rights': ['finance', 'public'],
            'description': 'Accès aux documents finance et publics'
        },
        {
            'name': '👔 Direction',
            'rights': ['finance', 'direction', 'public', 'admin'],
            'description': 'Accès étendu à plusieurs niveaux'
        },
        {
            'name': '🔒 Admin Système',
            'rights': ['admin', 'system', 'public', 'finance', 'direction'],
            'description': 'Accès maximal'
        }
    ]
    
    # Requête de test
    test_query = "configuration"  # Modifiez selon vos données
    
    results = []
    
    for profile in user_profiles:
        print(f"\n{profile['name']}:")
        print(f"📋 {profile['description']}")
        
        if profile['rights']:
            rights_str = ", ".join(profile['rights'])
            print(f"🔑 Droits: {rights_str}")
        
        # Recherche rapide pour compter
        search_results = indexer.search(test_query, profile['rights'], size=1)
        
        if search_results and 'hits' in search_results:
            total = search_results['hits']['total']['value']
            print(f"📊 Résultats accessibles: {total}")
            results.append({'profile': profile['name'], 'total': total, 'rights': profile['rights']})
        else:
            print(f"❌ Aucun résultat")
            results.append({'profile': profile['name'], 'total': 0, 'rights': profile['rights']})
    
    # Résumé comparatif
    print(f"\n📈 RÉSUMÉ COMPARATIF pour '{test_query}':")
    print("=" * 50)
    
    results.sort(key=lambda x: x['total'], reverse=True)
    
    for result in results:
        print(f"{result['profile']}: {result['total']} résultats")
    
    return results

# Test de comparaison avec/sans droits
print("🔐 Test de recherche avec droits:")
print("=" * 50)

# DÉCOMMENTEZ POUR TESTER (adaptez la requête à vos données):
# compare_search_with_without_rights("budget", ["finance", "public"])

print("⚠️ Tests désactivés - Décommentez et adaptez selon vos données")
print()

# Test des profils utilisateur
# test_different_user_profiles()

print("💡 Décommentez les fonctions ci-dessus pour tester selon vos documents")

## 🎯 Recherche Avancée et Analyse

In [ ]:
def analyze_search_patterns(queries: List[str], user_rights: List[str] = None) -> Dict:
    """Analyser les patterns de recherche sur plusieurs requêtes"""
    
    print(f"📊 Analyse de patterns de recherche:")
    print(f"🔍 {len(queries)} requêtes à analyser")
    
    if user_rights:
        rights_str = ", ".join(user_rights)
        print(f"👤 Droits utilisateur: {rights_str}")
    else:
        print(f"🌐 Recherche libre")
    
    print("=" * 60)
    
    analysis = {
        'queries': [],
        'total_searches': len(queries),
        'successful_searches': 0,
        'total_results': 0,
        'avg_results': 0,
        'avg_search_time': 0,
        'documents_found': set(),
        'score_distribution': []
    }
    
    total_time = 0
    
    for i, query in enumerate(queries, 1):
        print(f"\n{i}. Recherche: '{query}'")
        
        start_time = time.time()
        results = indexer.search(query, user_rights, size=10)
        search_time = time.time() - start_time
        
        total_time += search_time
        
        if results and 'hits' in results:
            hits = results['hits']['hits']
            total = results['hits']['total']['value']
            
            print(f"   📊 {total} résultats en {search_time:.3f}s")
            
            if total > 0:
                analysis['successful_searches'] += 1
                analysis['total_results'] += total
                
                # Analyser les scores
                scores = [hit['_score'] for hit in hits]
                max_score = max(scores) if scores else 0
                min_score = min(scores) if scores else 0
                avg_score = sum(scores) / len(scores) if scores else 0
                
                print(f"   🏆 Scores: max={max_score:.2f}, min={min_score:.2f}, moy={avg_score:.2f}")
                
                # Collecter les documents uniques
                for hit in hits:
                    filepath = hit['_source'].get('filepath', 'unknown')
                    analysis['documents_found'].add(filepath)
                
                analysis['score_distribution'].extend(scores)
            else:
                print(f"   ❌ Aucun résultat")
        else:
            print(f"   ❌ Erreur de recherche")
        
        query_result = {
            'query': query,
            'total_results': total if 'total' in locals() else 0,
            'search_time': search_time,
            'success': total > 0 if 'total' in locals() else False
        }
        analysis['queries'].append(query_result)
    
    # Calculer les moyennes
    if analysis['successful_searches'] > 0:
        analysis['avg_results'] = analysis['total_results'] / analysis['successful_searches']
    
    analysis['avg_search_time'] = total_time / len(queries)
    analysis['unique_documents'] = len(analysis['documents_found'])
    
    # Afficher le résumé
    print(f"\n📈 RÉSUMÉ D'ANALYSE:")
    print("=" * 50)
    print(f"🔍 Recherches effectuées: {analysis['total_searches']}")
    print(f"✅ Recherches fructueuses: {analysis['successful_searches']} ({analysis['successful_searches']/analysis['total_searches']*100:.1f}%)")
    print(f"📊 Total résultats: {analysis['total_results']}")
    print(f"📈 Moyenne par recherche: {analysis['avg_results']:.1f} résultats")
    print(f"⏱️ Temps moyen: {analysis['avg_search_time']:.3f}s")
    print(f"📚 Documents uniques trouvés: {analysis['unique_documents']}")
    
    if analysis['score_distribution']:
        max_score = max(analysis['score_distribution'])
        min_score = min(analysis['score_distribution'])
        avg_score = sum(analysis['score_distribution']) / len(analysis['score_distribution'])
        print(f"🏆 Scores globaux: max={max_score:.2f}, min={min_score:.2f}, moy={avg_score:.2f}")
    
    return analysis

def search_by_file_type(query: str, file_types: List[str] = None, user_rights: List[str] = None) -> Dict:
    """Recherche filtrée par type de fichier"""
    
    if file_types is None:
        file_types = ['.pdf', '.docx', '.txt']
    
    print(f"📁 Recherche par type de fichier: '{query}'")
    print(f"📋 Types: {', '.join(file_types)}")
    
    if user_rights:
        rights_str = ", ".join(user_rights)
        print(f"👤 Droits: {rights_str}")
    
    print("=" * 60)
    
    # Recherche standard d'abord
    all_results = indexer.search(query, user_rights, size=50)
    
    if not all_results or 'hits' not in all_results:
        print(f"❌ Aucun résultat pour '{query}'")
        return {}
    
    # Filtrer par type de fichier
    type_results = {}
    
    for file_type in file_types:
        type_results[file_type] = {'hits': [], 'count': 0}
    
    type_results['other'] = {'hits': [], 'count': 0}
    
    for hit in all_results['hits']['hits']:
        source = hit['_source']
        file_type = source.get('file_type', '').lower()
        
        if file_type in file_types:
            type_results[file_type]['hits'].append(hit)
            type_results[file_type]['count'] += 1
        else:
            type_results['other']['hits'].append(hit)
            type_results['other']['count'] += 1
    
    # Afficher les résultats par type
    for file_type, data in type_results.items():
        if data['count'] > 0:
            icon = "📄" if file_type == '.pdf' else "📝" if file_type in ['.docx', '.txt'] else "📁"
            print(f"\n{icon} Type {file_type}: {data['count']} résultats")
            
            for i, hit in enumerate(data['hits'][:3], 1):  # Limiter à 3 par type
                source = hit['_source']
                score = hit['_score']
                title = source.get('title', 'Sans titre')
                print(f"   {i}. {title} (score: {score:.2f})")
            
            if len(data['hits']) > 3:
                print(f"   ... et {len(data['hits']) - 3} autres")
    
    return type_results

# Exemple d'analyse de patterns
print(f"📊 Analyse avancée de recherche:")
print("=" * 50)

# Requêtes de test (adaptez selon vos données)
test_queries = [
    "configuration",
    "sécurité", 
    "télétravail",
    "budget",
    "procedure"
]

print(f"💡 Requêtes de test préparées: {len(test_queries)}")
print(f"⚠️ Tests désactivés - Décommentez pour analyser")
print()

# DÉCOMMENTEZ POUR ANALYSER:
# analysis = analyze_search_patterns(test_queries)
# 
# # Test avec droits
# print("\n" + "="*60)
# analysis_with_rights = analyze_search_patterns(test_queries, ["finance", "public"])

# Test de recherche par type de fichier
# search_by_file_type("configuration", ['.pdf', '.txt'], ["public"])

print("💡 Modifiez 'test_queries' selon vos documents indexés")

## 🎮 Interface de Recherche Interactive

In [ ]:
def interactive_search_session():
    """Session de recherche interactive avec options avancées"""
    
    print("🎮 Session de Recherche Interactive Avancée")
    print("=" * 60)
    print("Commandes disponibles:")
    print("  - Tapez votre recherche normale")
    print("  - 'rights [finance,public]' pour définir vos droits")
    print("  - 'size 10' pour changer le nombre de résultats")
    print("  - 'stats' pour voir les statistiques de l'index")
    print("  - 'clear' pour effacer les droits")
    print("  - 'quit' pour sortir")
    print("=" * 60)
    
    # État de la session
    session_state = {
        'user_rights': None,
        'results_size': 5,
        'search_count': 0,
        'total_results': 0
    }
    
    def show_current_config():
        print(f"\n⚙️ Configuration actuelle:")
        if session_state['user_rights']:
            rights_str = ", ".join(session_state['user_rights'])
            print(f"   🔑 Droits: {rights_str}")
        else:
            print(f"   🌐 Recherche libre (tous documents)")
        print(f"   📋 Taille des résultats: {session_state['results_size']}")
        print(f"   📊 Recherches effectuées: {session_state['search_count']}")
        print()
    
    show_current_config()
    
    while True:
        try:
            user_input = input("🔍 Votre commande: ").strip()
            
            if not user_input:
                continue
            
            # Commandes spéciales
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Session terminée !")
                print(f"📊 Résumé: {session_state['search_count']} recherches, {session_state['total_results']} résultats au total")
                break
            
            elif user_input.lower().startswith('rights '):
                # Définir les droits: rights finance,public,admin
                rights_str = user_input[7:].strip()
                if rights_str:
                    new_rights = [r.strip() for r in rights_str.split(',') if r.strip()]
                    session_state['user_rights'] = new_rights
                    rights_display = ", ".join(new_rights)
                    print(f"✅ Droits définis: {rights_display}")
                else:
                    print(f"❌ Format: rights finance,public,admin")
            
            elif user_input.lower() == 'clear':
                session_state['user_rights'] = None
                print(f"✅ Droits effacés - recherche libre activée")
            
            elif user_input.lower().startswith('size '):
                # Changer la taille: size 10
                try:
                    new_size = int(user_input[5:].strip())
                    if 1 <= new_size <= 50:
                        session_state['results_size'] = new_size
                        print(f"✅ Taille des résultats: {new_size}")
                    else:
                        print(f"❌ Taille doit être entre 1 et 50")
                except ValueError:
                    print(f"❌ Format: size 10")
            
            elif user_input.lower() == 'stats':
                # Afficher les stats de l'index
                stats = indexer.get_index_stats()
                if stats:
                    print(f"\n📊 Statistiques de l'index '{INDEX_NAME}':")
                    print(f"   📄 Documents: {stats.get('documents_count', 0)}")
                    print(f"   💾 Taille: {stats.get('size_mb', 0)} MB")
                else:
                    print(f"❌ Impossible d'obtenir les statistiques")
            
            elif user_input.lower() == 'config':
                show_current_config()
            
            else:
                # Recherche normale
                print(f"\n🔍 Recherche: '{user_input}'")
                result = search_and_display(
                    user_input, 
                    session_state['user_rights'], 
                    session_state['results_size'],
                    show_details=False
                )
                
                session_state['search_count'] += 1
                session_state['total_results'] += result.get('total', 0)
            
        except (KeyboardInterrupt, EOFError):
            print("\n👋 Session interrompue !")
            break
        except Exception as e:
            print(f"❌ Erreur: {e}")

# Interface interactive
print(f"🎮 Interface de recherche interactive:")
print("=" * 50)
print(f"⚠️ Interface désactivée pour éviter les blocages")
print(f"⚠️ Décommentez la ligne ci-dessous pour lancer:")
print()

# DÉCOMMENTEZ POUR LANCER L'INTERFACE INTERACTIVE:
# interactive_search_session()

print(f"💡 L'interface permet de:")
print(f"   - Rechercher avec des droits variables")
print(f"   - Ajuster la taille des résultats")
print(f"   - Voir les statistiques en temps réel")
print(f"   - Tester différents scénarios")

## ⚡ Tests de Performance

In [ ]:
def benchmark_search_performance(queries: List[str], iterations: int = 3) -> Dict:
    """Benchmarker les performances de recherche"""
    
    print(f"⚡ Benchmark de performance de recherche")
    print(f"🔍 {len(queries)} requêtes x {iterations} itérations = {len(queries) * iterations} tests")
    print("=" * 60)
    
    benchmark_results = {
        'queries': queries,
        'iterations': iterations,
        'total_searches': 0,
        'total_time': 0,
        'avg_time_per_search': 0,
        'min_time': float('inf'),
        'max_time': 0,
        'query_results': []
    }
    
    all_times = []
    
    for query in queries:
        print(f"\n🔍 Test: '{query}'")
        
        query_times = []
        query_results = []
        
        for i in range(iterations):
            start_time = time.time()
            results = indexer.search(query, user_rights=None, size=10)
            search_time = time.time() - start_time
            
            query_times.append(search_time)
            all_times.append(search_time)
            
            total_results = 0
            if results and 'hits' in results:
                total_results = results['hits']['total']['value']
            
            query_results.append({
                'iteration': i + 1,
                'time': search_time,
                'results_count': total_results
            })
            
            benchmark_results['total_searches'] += 1
            
            print(f"   Itération {i+1}: {search_time:.3f}s ({total_results} résultats)")
        
        # Statistiques par requête
        avg_time = sum(query_times) / len(query_times)
        min_time = min(query_times)
        max_time = max(query_times)
        
        print(f"   📊 Moyenne: {avg_time:.3f}s | Min: {min_time:.3f}s | Max: {max_time:.3f}s")
        
        benchmark_results['query_results'].append({
            'query': query,
            'avg_time': avg_time,
            'min_time': min_time,
            'max_time': max_time,
            'iterations': query_results
        })
    
    # Statistiques globales
    benchmark_results['total_time'] = sum(all_times)
    benchmark_results['avg_time_per_search'] = benchmark_results['total_time'] / benchmark_results['total_searches']
    benchmark_results['min_time'] = min(all_times)
    benchmark_results['max_time'] = max(all_times)
    
    # Afficher le résumé
    print(f"\n⚡ RÉSUMÉ DU BENCHMARK:")
    print("=" * 50)
    print(f"🔍 Total recherches: {benchmark_results['total_searches']}")
    print(f"⏱️ Temps total: {benchmark_results['total_time']:.3f}s")
    print(f"📊 Temps moyen: {benchmark_results['avg_time_per_search']:.3f}s")
    print(f"🏃 Plus rapide: {benchmark_results['min_time']:.3f}s")
    print(f"🐌 Plus lente: {benchmark_results['max_time']:.3f}s")
    
    # Évaluation des performances
    avg_ms = benchmark_results['avg_time_per_search'] * 1000
    
    if avg_ms < 50:
        perf_rating = "🚀 Excellent (< 50ms)"
    elif avg_ms < 100:
        perf_rating = "✅ Très bon (< 100ms)"
    elif avg_ms < 200:
        perf_rating = "👍 Bon (< 200ms)"
    elif avg_ms < 500:
        perf_rating = "⚠️ Acceptable (< 500ms)"
    else:
        perf_rating = "🐌 Lent (> 500ms)"
    
    print(f"🎯 Performance: {perf_rating}")
    
    return benchmark_results

def compare_rights_performance(query: str, iterations: int = 5) -> Dict:
    """Comparer les performances avec et sans droits"""
    
    print(f"⚖️ Comparaison performance avec/sans droits")
    print(f"🔍 Requête: '{query}' x {iterations} itérations")
    print("=" * 60)
    
    # Test sans droits
    print(f"🌐 Test SANS droits (recherche libre):")
    free_times = []
    
    for i in range(iterations):
        start_time = time.time()
        results_free = indexer.search(query, user_rights=None, size=10)
        search_time = time.time() - start_time
        free_times.append(search_time)
        
        total_free = results_free['hits']['total']['value'] if results_free and 'hits' in results_free else 0
        print(f"   Itération {i+1}: {search_time:.3f}s ({total_free} résultats)")
    
    # Test avec droits
    test_rights = ["finance", "public", "admin"]
    print(f"\n🔐 Test AVEC droits {test_rights}:")
    rights_times = []
    
    for i in range(iterations):
        start_time = time.time()
        results_rights = indexer.search(query, user_rights=test_rights, size=10)
        search_time = time.time() - start_time
        rights_times.append(search_time)
        
        total_rights = results_rights['hits']['total']['value'] if results_rights and 'hits' in results_rights else 0
        print(f"   Itération {i+1}: {search_time:.3f}s ({total_rights} résultats)")
    
    # Analyse comparative
    avg_free = sum(free_times) / len(free_times)
    avg_rights = sum(rights_times) / len(rights_times)
    
    print(f"\n📊 COMPARAISON:")
    print("=" * 40)
    print(f"🌐 Sans droits: {avg_free:.3f}s (moyenne)")
    print(f"🔐 Avec droits: {avg_rights:.3f}s (moyenne)")
    
    diff_ms = (avg_rights - avg_free) * 1000
    diff_percent = (avg_rights - avg_free) / avg_free * 100 if avg_free > 0 else 0
    
    if diff_ms > 0:
        print(f"⏱️ Surcharge droits: +{diff_ms:.1f}ms ({diff_percent:+.1f}%)")
    else:
        print(f"⚡ Avec droits plus rapide: {abs(diff_ms):.1f}ms ({diff_percent:+.1f}%)")
    
    return {
        'query': query,
        'free_avg': avg_free,
        'rights_avg': avg_rights,
        'overhead_ms': diff_ms,
        'overhead_percent': diff_percent
    }

# Tests de performance
print(f"⚡ Tests de performance:")
print("=" * 50)

# Requêtes pour benchmark
perf_queries = ["configuration", "sécurité", "budget"]

print(f"💡 Requêtes de test: {perf_queries}")
print(f"⚠️ Tests désactivés - Décommentez pour benchmarker")
print()

# DÉCOMMENTEZ POUR BENCHMARKER:
# benchmark_results = benchmark_search_performance(perf_queries, iterations=3)
# 
# # Test de comparaison avec/sans droits
# print("\n" + "="*60)
# rights_comparison = compare_rights_performance("configuration", iterations=5)

print("💡 Les tests mesurent:")
print("   - Temps de réponse moyen")
print("   - Variabilité des performances")
print("   - Impact du filtrage par droits")
print("   - Évaluation qualitative")

## 📋 Résumé - Recherche Avancée

✅ **Fonctionnalités disponibles dans ce notebook :**

### 🔍 **Recherche de Base**
- `search_and_display(query, user_rights, size, show_details)` - Recherche avec affichage détaillé
- Support du surlignage des termes trouvés
- Mesure automatique du temps de recherche
- Affichage des scores de pertinence

### 🔐 **Recherche avec Droits**
- `compare_search_with_without_rights(query, user_rights)` - Comparaison filtrée/libre
- `test_different_user_profiles()` - Test de différents profils utilisateur
- Analyse de l'impact du filtrage par droits
- Statistiques comparatives

### 📊 **Analyse Avancée**
- `analyze_search_patterns(queries, user_rights)` - Analyse multi-requêtes
- `search_by_file_type(query, file_types, user_rights)` - Recherche par type de fichier
- Distribution des scores de pertinence
- Identification des documents uniques trouvés

### 🎮 **Interface Interactive**
- `interactive_search_session()` - Session de recherche complète
- Commandes: `rights`, `size`, `stats`, `clear`, `config`
- Configuration dynamique des droits
- Historique des recherches

### ⚡ **Tests de Performance**
- `benchmark_search_performance(queries, iterations)` - Benchmark complet
- `compare_rights_performance(query, iterations)` - Impact des droits sur les performances
- Métriques: temps moyen, min/max, évaluation qualitative
- Mesure de la surcharge du filtrage par droits

### 💡 **Insights et Métriques**
- Taux de succès des recherches
- Nombre moyen de résultats par requête
- Documents uniques découverts
- Impact des droits sur les résultats
- Évaluation des performances (Excellent < 50ms, Bon < 200ms, etc.)

### 🎯 **Cas d'Usage Couverts**
- **Développement** : Tests et validation des fonctionnalités
- **Débogage** : Analyse des résultats inattendus
- **Optimisation** : Identification des goulots d'étranglement
- **Formation** : Démonstration des capacités de recherche
- **Monitoring** : Suivi des performances en conditions réelles

🚀 **Le système de recherche avancée avec droits optionnels est entièrement opérationnel !**

**💡 Prochaines étapes recommandées :**
1. Adaptez les requêtes de test à vos données réelles
2. Testez différents profils utilisateur selon votre organisation
3. Benchmarkez avec votre volume de données réel
4. Intégrez les fonctions dans votre application AskMe